# 🧬 JORINOVA NEXUS — Histopathology (tissue) CLASSIFIER (LC25000)

Classifies an H&E tissue patch as **normal tissue / adenocarcinoma / squamous cell carcinoma**.
Classification model (`yolov8m-cls`); metric = **top-1 accuracy**.

Produces `histology.pt`; the predicted class resolves to its finding via
`backend/ai_services/histology_findings.json`.

> **Data source: KAGGLE** — `andrewmvd/lung-and-colon-cancer-histopathological-images`
> (**LC25000**: 25,000 real 768px H&E patches, pathologist-validated). Upload `kaggle.json` in step 4.

**Before you start:** Runtime -> **Change runtime type** -> **T4 GPU**.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU, then rerun'

## 2. Install training dependencies

In [ ]:
%pip -q install ultralytics kaggle
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. Mount Drive (checkpoints survive a runtime reset)

In [ ]:
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/nexus_histology', exist_ok=True)
    print('Drive ready: /content/drive/MyDrive/nexus_histology')
except Exception as e:
    print('Drive not mounted (optional):', e)

## 4. Get the training data (KAGGLE — LC25000 lung+colon histopathology)
Upload your `kaggle.json` (kaggle.com → Settings → API → **Create New Token**). Downloads
**`andrewmvd/lung-and-colon-cancer-histopathological-images`** (5 classes) and folds them into the
`histology_findings.json` keys: benign→`normal_tissue`, aca→`adenocarcinoma`, scc→`squamous_cell_carcinoma`.

In [ ]:
# ── 4. Histology data: KAGGLE LC25000 (H&E lung+colon) -> train/val class folders ──
# andrewmvd/lung-and-colon-cancer-histopathological-images: 25k 768px patches, 5 classes
# (colon_aca, colon_bnt, lung_aca, lung_bnt, lung_scc) -> folded to histology_findings.json keys.
import os, glob, shutil, subprocess, sys, random
if not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    print('Upload kaggle.json (kaggle.com -> Settings -> API -> Create New Token) ...'); up = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    name = 'kaggle.json' if os.path.exists('kaggle.json') else list(up)[0]
    shutil.move(name, '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)

RAW = '/content/data/histo_raw'
if os.path.exists(RAW): shutil.rmtree(RAW)
os.makedirs(RAW, exist_ok=True)
rc = subprocess.run(['kaggle', 'datasets', 'download', '-d', 'andrewmvd/lung-and-colon-cancer-histopathological-images', '-p', RAW, '--unzip']).returncode
assert rc == 0, 'Kaggle download failed — accept the dataset rules on kaggle.com + check kaggle.json.'

def canon(n):
    n = n.lower()
    if 'scc' in n or 'squamous' in n: return 'squamous_cell_carcinoma'
    if 'aca' in n or 'adeno' in n: return 'adenocarcinoma'
    if 'bnt' in n or 'benign' in n or 'normal' in n: return 'normal_tissue'
    return None
def cls_of(path):
    for part in reversed(path.replace('\\', '/').split('/')):
        k = canon(part)
        if k: return k
    return None

exts = ('*.jpeg', '*.jpg', '*.png', '*.tif', '*.tiff', '*.bmp')
imgs = [p for e in exts for p in glob.glob(RAW + '/**/' + e, recursive=True)]
assert imgs, 'No images under ' + RAW
CLS = '/content/data/histo_cls'
if os.path.exists(CLS): shutil.rmtree(CLS)
buckets = {}
for p in imgs:
    k = cls_of(p)
    if k: buckets.setdefault(k, []).append(p)
assert len(buckets) >= 2, 'Fewer than 2 classes recognised — print imgs[:5] and send me the paths.'
for k, ps in buckets.items():
    random.shuffle(ps); nval = max(1, len(ps) // 6)
    for i, p in enumerate(ps):
        split = 'val' if i < nval else 'train'
        out = os.path.join(CLS, split, k); os.makedirs(out, exist_ok=True)
        shutil.copy(p, os.path.join(out, '%s_%d%s' % (k, i, os.path.splitext(p)[1])))
DATA_DIR = CLS
counts = {s: {c: len(os.listdir(os.path.join(CLS, s, c))) for c in sorted(os.listdir(os.path.join(CLS, s)))}
          for s in ('train', 'val') if os.path.isdir(os.path.join(CLS, s))}
print('DATA_DIR =', DATA_DIR); print('classes:', sorted(buckets)); print('counts:', counts)

## 5. Choose base weights (fine-tune, never from scratch)

In [ ]:
BASE_WEIGHTS = 'yolov8m-cls.pt'   # classification model (yolov8s-cls = faster/less accurate)
print('fine-tuning (classification) from:', BASE_WEIGHTS)

## 6. Fine-tune the classifier
Metric is **top-1 accuracy** (not mAP). Tissue patches → `imgsz=384` (texture detail),
`yolov8m-cls`, ~120 epochs, cosine LR + light dropout/erasing. Drop `batch` to 16 if you hit
CUDA OOM. H&E stain varies between labs — HSV augmentation helps generalisation.

In [ ]:
from ultralytics import YOLO
import os
PROJECT = '/content/drive/MyDrive/nexus_histology/runs'
try: os.makedirs(PROJECT, exist_ok=True)
except Exception: PROJECT = '/content/runs/classify'
model = YOLO(BASE_WEIGHTS)   # yolov8m-cls -> classification (patch-labelled histopathology)
model.train(data=DATA_DIR, epochs=120, imgsz=384, batch=32,
            project=PROJECT, name='histology', pretrained=True, patience=30, plots=True,
            cos_lr=True, dropout=0.2, fliplr=0.5, flipud=0.5, degrees=180,
            hsv_h=0.02, hsv_s=0.7, hsv_v=0.4, erasing=0.2)
m = model.val(); print(f'top1 = {m.top1:.3f}   top5 = {m.top5:.3f}')

## 6b. RESUME if a run disconnects (run INSTEAD of step 6)

In [ ]:
import os, glob
from ultralytics import YOLO
runs = sorted(glob.glob('/content/drive/MyDrive/nexus_histology/runs/**/weights/last.pt', recursive=True), key=os.path.getmtime)
loose = [p for p in sorted(glob.glob('/content/drive/MyDrive/nexus_histology/**/*.pt', recursive=True)
                           + glob.glob('/content/*.pt'), key=os.path.getmtime)
         if 'yolov8' not in os.path.basename(p).lower()]
assert runs or loose, 'No checkpoint yet — run step 6 first.'
ckpt = (runs or loose)[-1]; print('checkpoint:', ckpt)
model = None
try:
    if ckpt in runs: model = YOLO(ckpt); model.train(resume=True)
    else: raise RuntimeError('loose')
except Exception as e:
    print('continue-from-weights:', str(e)[:80]); model = None
if model is None:
    assert 'DATA_DIR' in globals(), 'Re-run step 4 first so DATA_DIR is set.'
    model = YOLO(ckpt); model.train(data=DATA_DIR, epochs=120, imgsz=384, batch=32,
        project='/content/drive/MyDrive/nexus_histology/runs', name='histology_cont', pretrained=True, patience=30,
        cos_lr=True, dropout=0.2, fliplr=0.5, flipud=0.5, degrees=180, erasing=0.2)
m = model.val(); print(f'top1 = {m.top1:.3f}   top5 = {m.top5:.3f}')

## 7. Export the model (Drive backup + download)

In [ ]:
import shutil, glob, os
DRIVE = '/content/drive/MyDrive/nexus_histology'
best = sorted([c for c in glob.glob('/content/**/weights/best.pt', recursive=True)
               + glob.glob(DRIVE + '/**/weights/best.pt', recursive=True) if os.path.isfile(c)],
              key=os.path.getmtime)[-1]
print('using weights:', best)
from ultralytics import YOLO; print('classes:', YOLO(best).model.names)
try: os.makedirs(DRIVE, exist_ok=True); shutil.copy(best, DRIVE + '/histology.pt'); print('Drive ->', DRIVE + '/histology.pt')
except Exception as e: print('drive copy skipped:', e)
from google.colab import files; files.download(best)

## 8. Put it in the app
1. Rename the download to **`histology.pt`** at **`backend/models/histology/histology.pt`**, commit, push.
2. Auto-loads for `image_type == "histology"`; the predicted class resolves via `backend/ai_services/histology_findings.json`. The vision service **detects classification models automatically** (reads top-1 + top-k, no code change).
3. On Render, build with `INSTALL_ML=1` (>=2 GB); else Claude vision reads the field.

> ⚠️ Screening aid only — a pathologist grades/stages and issues the diagnosis. If your dataset's class names differ from the map keys, send me the printed `classes:` list and I'll add aka-aliases.